### Open pycmap API

In [ ]:
import pycmap
from dotenv import load_dotenv
import os

# grab API key from hidden file
load_dotenv()
api_key = os.getenv("API_KEY")  
# load api key from CMAP
api = pycmap.API(token=api_key)

### 1. Load Seaflow dataset 
- Version 1.6 ([Ribalet et al. 2024](https://zenodo.org/records/10896099))
### 2. Perform Size Fractionation
- Size fractionate SeaFlow dataset based on cell size. Population-specific productivity as measured by 14C are typically size fractionated by 0.2 - 2µm or 0.2 - 3µm filters. 
- Options: 
    - "all", 
    - 2µm 
    - 3µm

In [25]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.simplefilter('ignore')

## prompt for size fractioning
sf = input("Perform size fractionation?")
# size fractionation
if sf.lower()[0] == 'y':
    # ask for size sf size
    max_size = int(input("What size to fractionate to?"))
    # reset to string for file naming
    max_size_str = f'{max_size}µm'
else:
    # set to all for file name saving
    max_size_str = 'all'

#### need different files for each sF version!!!
sf_file = f'../data/sf_clean_wide_{max_size_str}.pickle'
# check if file exists
if os.path.exists(sf_file):
    # read in file
    sf_clean = pd.read_pickle(sf_file)
else:
    new_sf=pd.read_parquet('../data/seaflow_1.6.parquet')
    ## cleaning
    # set tz to UTC
    new_sf['time']=pd.DatetimeIndex(new_sf['time']).tz_localize('UTC')
    # round to every min (don't need seconds resolution)
    new_sf['time'] = new_sf['time'].dt.round(freq='min')
    new_sf['day_year']=new_sf['time'].apply(lambda x: x.timetuple().tm_yday)
    ## define rows that are within Station ALOHA region
    new_sf['ALOHA']=False
    new_sf.loc[(new_sf['lat'] >= 22.25) &
            (new_sf['lat'] <= 23.25) & 
            (new_sf['lon'] <= -157.5) &
            (new_sf['lon'] >= -158.5), 'ALOHA']=True
    # fix longitude so it doesn't get messed up by IDL
    new_sf['lon']=np.where(new_sf['lon']>0, new_sf['lon'], new_sf['lon']+360)
    
    # size fractionation
    if sf.lower()[0] == 'y':
        # size fractionate based on input size
        diam_cols = [col for col in new_sf.columns if 'diam' in col]
        new_sf[diam_cols].mask((new_sf[diam_cols] < 0.2) | (new_sf[diam_cols] >= max_size), np.nan)

    # round to every hour for each cruise
    hourly_sf = new_sf.groupby(['cruise', pd.Grouper(key = 'time', freq = '1H')]).mean().reset_index()

    ## fill in missing lat/lons from cruise trajectories using CMAP
    dfs = []
    ## grab the tracks from CMAP and round to every hour
    for cruise in hourly_sf['cruise'].unique():
        print(cruise)
        # resample such that missing data is listed as nan 
        cruise_seaflow = hourly_sf.loc[hourly_sf['cruise'] == cruise]
        start_time = np.min(cruise_seaflow['time'])
        end_time = np.max(cruise_seaflow['time'])

        # edge case for instruments
        if ('_740' in cruise) | ('_130' in cruise) | ('751' in cruise):
            cruise = cruise.split('_')[0]
        # pull cruise track from cmap
        try:
            cruise_track = api.cruise_trajectory(cruise)
        except:
            # if not found by api, just use original cruise dataset from seaflow
            dfs.append(cruise_seaflow)
            continue
        # if no error, continue in loop
        cruise_track['lon']=np.where(cruise_track['lon']>0, cruise_track['lon'], cruise_track['lon']+360)
        cruise_track['time']=pd.DatetimeIndex(cruise_track['time']).tz_localize('UTC')
        # round to every hour
        hourly_track = cruise_track.groupby([pd.Grouper(key = 'time', freq = '1H')]).mean().reset_index()
        hourly_track['cruise'] = cruise

        
        # subset track by the start and end times
        track_subset = hourly_track.loc[(hourly_track['time'] >= start_time) & 
                    (hourly_track['time'] <= end_time)].reset_index()
        # first concatenate, then group by time and agg lat and lon
        cruise_concat = pd.concat([cruise_seaflow, hourly_track])
        cruise_agg = cruise_concat.groupby(['time', 'cruise']).mean().reset_index()
        # save into list of dfs
        dfs.append(cruise_agg)
    # concatenate all dfs together
    sf_clean_wide = pd.concat(dfs)
    # save into file
    sf_clean_wide.to_pickle(f'../data/sf_clean_wide_{max_size_str}.pickle')

#### Check for seasonal variation in sampling

In [26]:
test_df = sf_clean_wide.copy()
test_df['month'] = test_df['time'].dt.month
test_df['season'] = test_df['month'] %12 // 3 + 1
seasons = {1: 'Winter', 2: 'Spring', 3: 'Summer', 4: 'Fall'}
test_df['season'] = test_df['season'].map(seasons)

grouped_df = test_df.groupby(['season']).count().reset_index().sort_values(by = 'time')

In [27]:
total_days = np.sum(grouped_df['time'])
for season in pd.unique(grouped_df['season']):
    sub_df = grouped_df.loc[grouped_df['season'] == season]
    print(f"{season} : {sub_df['time'].values[0] / total_days}")

Winter : 0.1823918420919784
Spring : 0.23978999444696855
Fall : 0.2659397243677116
Summer : 0.3118784390933414


### Reformat from wide to long

In [28]:
sf_file_long = f'../data/sf_clean_long_{max_size_str}.pickle'
# check if file exists
if os.path.exists(sf_file_long):
        # read in file
        sf_clean = pd.read_pickle(sf_file_long)
else:
        ## melt to long
        #double melt, first by abundance
        temp_sf=sf_clean_wide.rename(columns={'abundance_prochloro':'prochloro',
                                        'abundance_synecho':'synecho',
                                        'abundance_picoeuk':'picoeuk'})
        abund_sf=pd.melt(temp_sf, id_vars=['time','day_year','cruise','lat','lon', 'ALOHA'],
                var_name='pop',
                value_vars=['prochloro', 'synecho','picoeuk'],
                value_name='abundance')
        # melt by Qc
        temp_sf=sf_clean_wide.rename(columns={'Qc_prochloro':'prochloro',
                                        'Qc_synecho':'synecho',
                                        'Qc_picoeuk':'picoeuk'})
        qc_sf=pd.melt(temp_sf, id_vars=['time','day_year','cruise','lat','lon', 'ALOHA'],
                var_name='pop',
                value_vars=['prochloro', 'synecho','picoeuk'],
                value_name='Qc_hour')
        # by diam
        temp_sf=sf_clean_wide.rename(columns={'diam_prochloro':'prochloro',
                                        'diam_synecho':'synecho',
                                        'diam_picoeuk':'picoeuk'})
        diam_sf=pd.melt(temp_sf, id_vars=['time','day_year','cruise','lat','lon', 'ALOHA'],
                var_name='pop',
                value_vars=['prochloro', 'synecho','picoeuk'],
                value_name='diam_hour')

        # merge Qc and abundance dfs
        sf_merge1=abund_sf.merge(qc_sf)
        # merge again with diam 
        sf_merge2=sf_merge1.merge(diam_sf)
        # remove abundance that's too low
        sf_clean=sf_merge2[sf_merge2['abundance']> 0.02]
        # save as file
        sf_clean.to_pickle(f'../data/sf_clean_long_{max_size_str}.pickle')

### Calculate day and night using solar calculator
- optional step, solar calculator runs again during TSD modeling step

In [29]:
# load modules from scripts folder, replace path with your own
full_path = '/Users/Kathy/Desktop/UW/seaflow/decomposition_project/scripts/'

from astral import Observer
import sys
sys.path.insert(0, full_path)
from diel_tools_clean import sunrise_sunset, label_daytime

# check if file exists, create hourly_sf if not
filepath = f'../data/hourly_sf_v1_{max_size_str}.pickle'

if os.path.exists(filepath):
    # open hourly_sf file
    sf_clean_diel = pd.read_pickle(filepath)
else:
    print(f"'{filepath}' does not exist. Creating file...")
    # create observer col
    sf_clean['obs']=sf_clean.apply(lambda x: Observer(x.lat, x.lon, 0), axis=1)

    # create hourly rounded sunrise/sunset cols for sf_clean 
    res=sf_clean.apply(lambda x: sunrise_sunset(x.time, x.obs), axis=1)
    sf_clean[['sunrise', 'sunset']]=pd.DataFrame(res.tolist(), index=sf_clean.index)
    # aggregate and choose one sunrise/sunset value (max value of time)
    sf_clean_diel=sf_clean.groupby([pd.Grouper(key='time',freq='1H'),
                    'pop','cruise']).agg({
        'lat':'mean',
        'lon':'mean',
        'ALOHA':'mean',
        'abundance':'mean',
        'Qc_hour':'mean',
        'sunrise':'max',
        'sunset':'max'
    }).reset_index()

    # apply to dataframe
    # annette has 8817 after filtering (8889 if no filtering) and excluding croco (i have 8874)
    sf_clean_diel['night']=sf_clean_diel.apply(lambda x: label_daytime(x.time, x.sunrise, x.sunset), axis=1)
    sf_clean_diel['par']=0
    # save data 
    sf_clean_diel.to_pickle(filepath)

'../data/hourly_sf_v1_all.pickle' does not exist. Creating file...


## Run STL and 3-Day Rolling models
- Run on prochloro, synecho, picoeuk populations

In [21]:
# load modules from scripts folder, replace path with your own
full_path = '../scripts/'
import sys
sys.path.insert(0, full_path)
from tsd_functions_clean import run_full_model

# load files from saved files if they exist
tsd_file = f'../data/all_seaflow_tsd_mixed_v1_{max_size_str}.pickle'
rates_file = f'../data/all_rates_mixed_v1_{max_size_str}.pickle'
fails_file = f'../data/failed_days_mixed_v1_{max_size_str}.pickle'
final_file = f'../data/final_rates_v1_{max_size_str}.pickle'

## all files must exist or run model
if (os.path.exists(tsd_file)) & (os.path.exists(rates_file)) & (os.path.exists(fails_file)) & (os.path.exists(final_file)):
    # open files
    all_tsd = pd.read_pickle(tsd_file)
    daily_rates = pd.read_pickle(rates_file)
    # get daily data
    mean_tsd=all_tsd.groupby(['cruise','cruise_day','pop'])[['lat','lon','pop']].mean().reset_index()
    # merge
    merge_rates=daily_rates.merge(mean_tsd)
    failed_days_df = pd.read_pickle(fails_file)
    # remove days that have failed to run from tsd df
    merge_tsd=all_tsd.merge(failed_days_df[['cruise','cruise_day','pop','model']], 
                        how='outer', indicator=True)
    good_tsd=merge_tsd[merge_tsd['_merge']=='left_only']
    final_rates=pd.read_pickle(final_file)
# run model for each cruise and population
else:
    # set par col to 0 if doesn't exist
    sf_clean['par'] = 0
    pops=['prochloro','synecho','picoeuk']
    # save dataframes
    growth_rates=[]
    tsd_results=[]
    impute_dfs = []
    all_names=pd.unique(sf_clean['cruise'])
    # cruises that fail to run due to being too short
    failed_cruises=[]
    # days that fail to run due to not having enough data points
    failed_days=[]
    for cruise_name in all_names:
        # subset by cruise
        print(cruise_name)
        cruise_df=sf_clean.loc[sf_clean['cruise']==cruise_name]
        for pop in pops:
            print(pop)
            # subset df by dataframe
            pop_df=cruise_df.loc[cruise_df['pop']==pop]
            # add column for missing data to be filled
            pop_df['data_with_missing']=pop_df['Qc_hour']
            # run full model (V1)
            impute_df, tsd_df, growth, skip_df=run_full_model(df=pop_df, 
                                            col='Qc_hour', missing_col='data_with_missing',pop=pop)
            # keep going if failed
            if tsd_df is None:
                # save pop and cruise
                failed_cruises.append({cruise_name:pop})
                continue
            # save results
            impute_dfs.append(impute_df)
            growth_rates.append(growth)
            tsd_results.append(tsd_df)
            failed_days.append(skip_df)

    # save data
    impute_results = pd.concat(impute_dfs)
    impute_results.to_pickle(f'../data/imputed_results_v1_{max_size_str}.pickle')

    all_tsd=pd.concat(tsd_results)
    # # get daily data
    mean_tsd=all_tsd.groupby(['cruise','cruise_day','pop'])[['lat','lon','pop', 'biomass']].mean().reset_index()
    daily_rates=pd.concat(growth_rates)
    # merge
    merge_rates=daily_rates.merge(mean_tsd)
    failed_days_df=pd.concat(failed_days)
    merge_tsd=all_tsd.merge(failed_days_df[['cruise','cruise_day','pop','model']], 
                        how='outer', indicator=True)
    good_tsd=merge_tsd[merge_tsd['_merge']=='left_only']
    ## save files
    all_tsd.to_pickle(f'../data/all_seaflow_tsd_mixed_v1_{max_size_str}.pickle')
    daily_rates.to_pickle(f'../data/all_rates_mixed_v1_{max_size_str}.pickle')
    failed_days_df.to_pickle(f'../data/failed_days_mixed_v1_{max_size_str}.pickle')

CN11ID
prochloro
synecho
picoeuk
CN12ID
prochloro
synecho
picoeuk
CN13ID
prochloro
synecho
picoeuk
FK180310-1
prochloro
synecho
picoeuk
FK180310-2
prochloro
synecho
picoeuk
KM1314
prochloro
synecho
picoeuk
KM1427
prochloro
synecho
picoeuk
KM1502
prochloro
synecho
picoeuk
KM1508
prochloro
synecho
picoeuk
KM1510
prochloro
Not enough data for imputation
synecho
Not enough data for imputation
picoeuk
Not enough data for imputation
KM1512
prochloro
synecho
picoeuk
KM1513
prochloro
synecho
picoeuk
KM1518
prochloro
synecho
picoeuk
KM1601
prochloro
synecho
picoeuk
KM1602
prochloro
synecho
picoeuk
KM1603
prochloro
Not enough data for imputation
synecho
Not enough data for imputation
picoeuk
Not enough data for imputation
KM1708
prochloro
synecho
picoeuk
KM1709
prochloro
synecho
picoeuk
KM1712
prochloro
synecho
picoeuk
KM1713
prochloro
synecho
picoeuk
KM1717
prochloro
synecho
picoeuk
KM1802
prochloro
synecho
picoeuk
KM1805
prochloro
synecho
picoeuk
KM1821
prochloro
synecho
picoeuk
KM1823
prochlo

## Debugging
- Remove files if you want to run the model again

In [10]:
# prompt to check if you want to run the model again
run_again = input("Would you like to rerun the model?")
if run_again.lower()[0] == 'y':
    # delete files to rerun (for testing)
    os.remove(tsd_file)
    os.remove(rates_file)
    os.remove(fails_file)
    os.remove(final_file)
else:
    print("Continue")

Continue


## Quality Control
Don't need to run this step if you already have the final rates file loaded

1) Daily growth has a significant (p < 0.01) slope
2) Model selection

### 1. Check for significance in daily growth

In [22]:
# distinguish between significant and non-significant rates
sig_rates=merge_rates.loc[(merge_rates['pval']<0.01)&(merge_rates['daily_growth']>0)]
bad_rates=merge_rates.loc[(merge_rates['pval']>0.01)|(merge_rates['daily_growth']<0)]

### 2. Model selection based on RMSE*

*This is also the same as taking the square root of the residual for each model

In [23]:
from model_qc import r2_rmse
# grab good days from tsd df
sum_sig_rates=sig_rates[['cruise','cruise_day','pop','model']]
sig_tsd=sum_sig_rates.merge(good_tsd)
# calculate the RMSE for each day and model
group_rmse=sig_tsd.groupby(['cruise','cruise_day','pop','model']).apply(r2_rmse).reset_index()
# get minimum rmse per model on each day
good_models=group_rmse.groupby(['cruise','cruise_day','pop']).agg({
    'rmse':'min'
}).reset_index()
# retrieve model infromation from first grouped df
good_models=good_models.merge(group_rmse)
# filter good rates from good models
final_rates=sig_rates.merge(good_models)
final_rates=final_rates.loc[final_rates['productivity']<20]
# save final rates
final_rates.to_pickle(f'../data/final_rates_v1_{max_size_str}.pickle')